In [3]:
import numpy as np

In [4]:
# one LSTM module from scratch using only numpy
class LSTM:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        # initialize weights and biases
        self.W_f = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_f = np.zeros((hidden_size, 1))
        self.W_i = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_i = np.zeros((hidden_size, 1))
        self.W_c = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_c = np.zeros((hidden_size, 1))
        self.W_o = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_o = np.zeros((hidden_size, 1))
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    def tanh(self, x):
        return np.tanh(x)
    def forward(self, x, h_prev, c_prev):
        # concatenate input and previous hidden state
        x_concat = np.concatenate((x, h_prev), axis=0)
        # forget gate
        f_t = self.sigmoid(np.dot(self.W_f, x_concat) + self.b_f)
        # input gate
        i_t = self.sigmoid(np.dot(self.W_i, x_concat) + self.b_i)
        # candidate value
        c_tilde = self.tanh(np.dot(self.W_c, x_concat) + self.b_c)
        # new cell state
        c_t = f_t * c_prev + i_t * c_tilde
        # output gate
        o_t = self.sigmoid(np.dot(self.W_o, x_concat) + self.b_o)
        # new hidden state
        h_t = o_t * self.tanh(c_t)
        return h_t, c_t
    

In [5]:
# try out the LSTM with a sequence of increasing numbers
if __name__ == "__main__":
    input_size = 1
    hidden_size = 5
    lstm = LSTM(input_size, hidden_size)
    sequence_length = 10
    x_sequence = np.array([[i] for i in range(sequence_length)])
    h_prev = np.zeros((hidden_size, 1))
    c_prev = np.zeros((hidden_size, 1))
    for t in range(sequence_length):
        x_t = x_sequence[t].reshape(-1, 1)
        h_prev, c_prev = lstm.forward(x_t, h_prev, c_prev)
        print(f"Time step {t}: h_t = {h_prev.ravel()}")

Time step 0: h_t = [0. 0. 0. 0. 0.]
Time step 1: h_t = [-0.00035903  0.00010684 -0.00135643 -0.00105287 -0.00251501]
Time step 2: h_t = [-0.00089295  0.00027285 -0.00340106 -0.00263035 -0.0063154 ]
Time step 3: h_t = [-0.00151228  0.00047145 -0.0057948  -0.00446567 -0.0107937 ]
Time step 4: h_t = [-0.00217351  0.00068795 -0.0083659  -0.00642424 -0.01564833]
Time step 5: h_t = [-0.00285545  0.0009143  -0.01102747 -0.00843822 -0.0207286 ]
Time step 6: h_t = [-0.00354777  0.0011461  -0.01373558 -0.01047341 -0.02595848]
Time step 7: h_t = [-0.00424545  0.00138096 -0.01646799 -0.01251254 -0.03129897]
Time step 8: h_t = [-0.00494603  0.00161758 -0.01921338 -0.01454685 -0.03672945]
Time step 9: h_t = [-0.00564831  0.00185522 -0.02196592 -0.01657185 -0.04223844]


## Train one LSTM module on `+1` sequences (from scratch, NumPy)
We train a **single LSTM layer** with a linear output head to predict the next value in a sequence where each step increases by 1.

For one time-step (with `z_t = [x_t; h_{t-1}]`):
- $f_t=\sigma(W_f z_t+b_f)$
- $i_t=\sigma(W_i z_t+b_i)$
- $\tilde{c}_t=\tanh(W_c z_t+b_c)$
- $c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t$
- $o_t=\sigma(W_o z_t+b_o)$
- $h_t=o_t\odot\tanh(c_t)$
- $\hat{y}_t=W_y h_t+b_y$

Loss: MSE over all sequence steps. Backpropagation uses BPTT through these equations.

In [8]:
# Trainable single-layer LSTM + linear head
import numpy as np

np.random.seed(42)

class TrainableLSTM:
    def __init__(self, input_size=1, hidden_size=8, output_size=1, lr=1e-2):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lr = lr

        scale = 0.1
        self.W_f = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.b_f = np.zeros((hidden_size, 1))
        self.W_i = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.b_i = np.zeros((hidden_size, 1))
        self.W_c = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.b_c = np.zeros((hidden_size, 1))
        self.W_o = np.random.randn(hidden_size, input_size + hidden_size) * scale
        self.b_o = np.zeros((hidden_size, 1))

        self.W_y = np.random.randn(output_size, hidden_size) * scale
        self.b_y = np.zeros((output_size, 1))

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    def forward_sequence(self, x_seq, h0=None, c0=None, store_cache=True):
        T = x_seq.shape[0]
        h_prev = np.zeros((self.hidden_size, 1)) if h0 is None else h0.copy()
        c_prev = np.zeros((self.hidden_size, 1)) if c0 is None else c0.copy()

        y_preds = []
        caches = []

        for t in range(T):
            x_t = x_seq[t].reshape(self.input_size, 1)
            z_t = np.vstack([x_t, h_prev])

            f_t = self.sigmoid(self.W_f @ z_t + self.b_f)
            i_t = self.sigmoid(self.W_i @ z_t + self.b_i)
            g_t = np.tanh(self.W_c @ z_t + self.b_c)
            c_t = f_t * c_prev + i_t * g_t
            o_t = self.sigmoid(self.W_o @ z_t + self.b_o)
            h_t = o_t * np.tanh(c_t)

            # Residual next-step head: predict increment on top of input
            y_t = x_t + (self.W_y @ h_t + self.b_y)
            y_preds.append(y_t)

            if store_cache:
                caches.append({
                    'x_t': x_t, 'z_t': z_t,
                    'h_prev': h_prev, 'c_prev': c_prev,
                    'f_t': f_t, 'i_t': i_t, 'g_t': g_t,
                    'c_t': c_t, 'o_t': o_t, 'h_t': h_t
                })

            h_prev, c_prev = h_t, c_t

        y_preds = np.stack(y_preds, axis=0).reshape(T, self.output_size, 1)
        return y_preds, (h_prev, c_prev), caches

    def backward_sequence(self, dy_seq, caches, clip=1.0):
        dW_f = np.zeros_like(self.W_f); db_f = np.zeros_like(self.b_f)
        dW_i = np.zeros_like(self.W_i); db_i = np.zeros_like(self.b_i)
        dW_c = np.zeros_like(self.W_c); db_c = np.zeros_like(self.b_c)
        dW_o = np.zeros_like(self.W_o); db_o = np.zeros_like(self.b_o)
        dW_y = np.zeros_like(self.W_y); db_y = np.zeros_like(self.b_y)

        dh_next = np.zeros((self.hidden_size, 1))
        dc_next = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(caches))):
            cache = caches[t]
            z_t = cache['z_t']
            c_prev = cache['c_prev']
            f_t = cache['f_t']; i_t = cache['i_t']; g_t = cache['g_t']
            c_t = cache['c_t']; o_t = cache['o_t']; h_t = cache['h_t']

            dy = dy_seq[t].reshape(self.output_size, 1)

            dW_y += dy @ h_t.T
            db_y += dy

            dh = self.W_y.T @ dy + dh_next

            tanh_c = np.tanh(c_t)
            do = dh * tanh_c
            do_pre = do * o_t * (1 - o_t)

            dc = dh * o_t * (1 - tanh_c ** 2) + dc_next

            df = dc * c_prev
            df_pre = df * f_t * (1 - f_t)

            di = dc * g_t
            di_pre = di * i_t * (1 - i_t)

            dg = dc * i_t
            dg_pre = dg * (1 - g_t ** 2)

            dW_f += df_pre @ z_t.T
            db_f += df_pre
            dW_i += di_pre @ z_t.T
            db_i += di_pre
            dW_c += dg_pre @ z_t.T
            db_c += dg_pre
            dW_o += do_pre @ z_t.T
            db_o += do_pre

            dz = (self.W_f.T @ df_pre +
                  self.W_i.T @ di_pre +
                  self.W_c.T @ dg_pre +
                  self.W_o.T @ do_pre)

            dh_next = dz[self.input_size:, :]
            dc_next = dc * f_t

        grads = [dW_f, db_f, dW_i, db_i, dW_c, db_c, dW_o, db_o, dW_y, db_y]
        for g in grads:
            np.clip(g, -clip, clip, out=g)

        self.W_f -= self.lr * dW_f; self.b_f -= self.lr * db_f
        self.W_i -= self.lr * dW_i; self.b_i -= self.lr * db_i
        self.W_c -= self.lr * dW_c; self.b_c -= self.lr * db_c
        self.W_o -= self.lr * dW_o; self.b_o -= self.lr * db_o
        self.W_y -= self.lr * dW_y; self.b_y -= self.lr * db_y

def make_increasing_sequence(start, length, step=1.0):
    return np.array([[start + step * t] for t in range(length)], dtype=np.float64)


In [9]:
def train_model(model_class, hyperparams, train_data, val_data=None, epochs=10, early_stopping_patience=None):
    """
    Trains a model with given hyperparameters and class.
    Args:
        model_class: The class of the model to instantiate.
        hyperparams: Dictionary of hyperparameters for the model.
        train_data: Training data (tuple or generator).
        val_data: Validation data (optional).
        epochs: Number of epochs to train.
        early_stopping_patience: Patience for early stopping (optional).
    Returns:
        model: The trained model.
        history: Training history (if available).
    """
    model = model_class(**hyperparams)
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    for epoch in range(epochs):
        train_loss = 0.0
        train_batches = 0
        for x, y_true in train_data:
            y_pred, _, caches = model.forward_sequence(x, store_cache=True)
            err = y_pred - y_true.reshape(x.shape[0], 1, 1)
            loss = np.mean(err ** 2)
            train_loss += loss
            train_batches += 1
            dy = (2.0 / x.shape[0]) * err
            model.backward_sequence(dy, caches, clip=1.0)
        train_loss /= max(train_batches, 1)
        history['train_loss'].append(train_loss)
        val_loss = None
        if val_data:
            val_loss = 0.0
            val_batches = 0
            for x_val, y_val_true in val_data:
                y_val_pred, _, _ = model.forward_sequence(x_val, store_cache=False)
                err_val = y_val_pred - y_val_true.reshape(x_val.shape[0], 1, 1)
                loss_val = np.mean(err_val ** 2)
                val_loss += loss_val
                val_batches += 1
            val_loss /= max(val_batches, 1)
            history['val_loss'].append(val_loss)
            if early_stopping_patience:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f"Early stopping at epoch {epoch+1}")
                        break
        if (epoch+1) % 5 == 0 or epoch == 0:
            msg = f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {train_loss:.8f}"
            if val_loss is not None:
                msg += f" | Val Loss: {val_loss:.8f}"
            print(msg)
    return model, history

In [18]:
# Hyperparameters (<50 epochs)
hidden_size = 1
lr = 1e-2
epochs = 5
seq_len = 20
batch_size = 12

# Training data generator for increasing sequences
def train_data_generator(batch_size, seq_len):
    for _ in range(batch_size):
        start = np.random.uniform(-400.0, 400.0)
        seq = make_increasing_sequence(start, seq_len + 1, step=1.5)
        x = seq[:-1]
        y_true = seq[1:]
        yield x, y_true

hyperparams = dict(input_size=1, hidden_size=hidden_size, output_size=1, lr=lr)

# Train using the generic train_model function
model, history = train_model(
    model_class=TrainableLSTM,
    hyperparams=hyperparams,
    train_data=train_data_generator(batch_size, seq_len),
    epochs=epochs
)

print("\nTraining complete.")

# Teacher-forced test
test_start = 7.0
test_seq = make_increasing_sequence(test_start, seq_len + 1, step=1.0)
x_test = test_seq[:-1]
y_test = test_seq[1:]

y_test_pred, _, _ = model.forward_sequence(x_test, store_cache=False)
y_pred_arr = y_test_pred.reshape(seq_len, 1)
y_true_arr = y_test.reshape(seq_len, 1)

print("\nTeacher-forced next-step predictions:")
for t in range(seq_len):
    print(f"input={x_test[t,0]:6.2f} -> pred_next={y_pred_arr[t,0]:8.4f} | true_next={y_true_arr[t,0]:6.2f}")

# Autoregressive rollout from one starting value
h_roll = np.zeros((hidden_size, 1))
c_roll = np.zeros((hidden_size, 1))
x_cur = np.array([[test_start]])

generated = [test_start]
for _ in range(10):
    y_one, (h_roll, c_roll), _ = model.forward_sequence(x_cur, h_roll, c_roll, store_cache=False)
    x_cur = y_one[-1].reshape(1, 1)
    generated.append(float(x_cur[0, 0]))

print("\nAutoregressive rollout (should increase by ~1):")
print([round(v, 4) for v in generated])

Epoch  1/5 | Train Loss: 1.99802324
Epoch  5/5 | Train Loss: 0.00000000

Training complete.

Teacher-forced next-step predictions:
input=  7.00 -> pred_next=  7.1283 | true_next=  8.00
input=  8.00 -> pred_next=  8.1325 | true_next=  9.00
input=  9.00 -> pred_next=  9.1351 | true_next= 10.00
input= 10.00 -> pred_next= 10.1372 | true_next= 11.00
input= 11.00 -> pred_next= 11.1391 | true_next= 12.00
input= 12.00 -> pred_next= 12.1410 | true_next= 13.00
input= 13.00 -> pred_next= 13.1428 | true_next= 14.00
input= 14.00 -> pred_next= 14.1447 | true_next= 15.00
input= 15.00 -> pred_next= 15.1466 | true_next= 16.00
input= 16.00 -> pred_next= 16.1485 | true_next= 17.00
input= 17.00 -> pred_next= 17.1504 | true_next= 18.00
input= 18.00 -> pred_next= 18.1523 | true_next= 19.00
input= 19.00 -> pred_next= 19.1542 | true_next= 20.00
input= 20.00 -> pred_next= 20.1561 | true_next= 21.00
input= 21.00 -> pred_next= 21.1580 | true_next= 22.00
input= 22.00 -> pred_next= 22.1599 | true_next= 23.00
input

In [ ]:
def make_exp_decay_sequence(start, length, decay=0.8):
    return np.array([[start * (decay ** t)] for t in range(length)], dtype=np.float64)

Epoch  1/40 | Loss: 0.01986212
Epoch  5/40 | Loss: 0.01540184
Epoch 10/40 | Loss: 0.01357585
Epoch 15/40 | Loss: 0.01132780
Epoch 20/40 | Loss: 0.00748251
Epoch 25/40 | Loss: 0.00387370
Epoch 30/40 | Loss: 0.00199106
Epoch 35/40 | Loss: 0.00120935
Epoch 40/40 | Loss: 0.00095955

Training complete.

Teacher-forced next-step predictions:
input=2.0000 -> pred_next=  1.6702 | true_next=1.6000
input=1.6000 -> pred_next=  1.2827 | true_next=1.2800
input=1.2800 -> pred_next=  1.0109 | true_next=1.0240
input=1.0240 -> pred_next=  0.8083 | true_next=0.8192
input=0.8192 -> pred_next=  0.6498 | true_next=0.6554
input=0.6554 -> pred_next=  0.5220 | true_next=0.5243
input=0.5243 -> pred_next=  0.4176 | true_next=0.4194
input=0.4194 -> pred_next=  0.3320 | true_next=0.3355

Autoregressive rollout (should decay by ~0.8):
[2.0, 1.6702, 1.3428, 1.0608, 0.8349, 0.6591, 0.5225, 0.4144, 0.3265, 0.2532, 0.1903]


In [ ]:
# Hyperparameters
hidden_size = 1
lr = 1e-2
epochs = 40
seq_len = 8
batch_size = 256
decay = 0.8

model = TrainableLSTM(input_size=1, hidden_size=hidden_size, output_size=1, lr=lr)

losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    for _ in range(batch_size):
        start = np.random.uniform(1.0, 3.0)
        seq = make_exp_decay_sequence(start, seq_len + 1, decay=decay)

        x = seq[:-1]
        y_true = seq[1:]

        y_pred, _, caches = model.forward_sequence(x, store_cache=True)

        err = y_pred - y_true.reshape(seq_len, 1, 1)
        loss = np.mean(err ** 2)
        epoch_loss += loss

        dy = (2.0 / seq_len) * err
        model.backward_sequence(dy, caches, clip=1.0)

    epoch_loss /= batch_size
    losses.append(epoch_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{epochs} | Loss: {epoch_loss:.8f}")

print("\nTraining complete.")

# Teacher-forced test
test_start = 2.0
test_seq = make_exp_decay_sequence(test_start, seq_len + 1, decay=decay)
x_test = test_seq[:-1]
y_test = test_seq[1:]

y_test_pred, _, _ = model.forward_sequence(x_test, store_cache=False)
y_pred_arr = y_test_pred.reshape(seq_len, 1)
y_true_arr = y_test.reshape(seq_len, 1)

print("\nTeacher-forced next-step predictions:")
for t in range(seq_len):
    print(f"input={x_test[t,0]:6.4f} -> pred_next={y_pred_arr[t,0]:8.4f} | true_next={y_true_arr[t,0]:6.4f}")

# Autoregressive rollout from one starting value
h_roll = np.zeros((hidden_size, 1))
c_roll = np.zeros((hidden_size, 1))
x_cur = np.array([[test_start]])

generated = [test_start]
for _ in range(10):
    y_one, (h_roll, c_roll), _ = model.forward_sequence(x_cur, h_roll, c_roll, store_cache=False)
    x_cur = y_one[-1].reshape(1, 1)
    generated.append(float(x_cur[0, 0]))

print("\nAutoregressive rollout (should decay by ~0.8):")
print([round(v, 4) for v in generated])

In [ ]:
# Train LSTM on slow sine sequence (3 periods in 100 steps)

def make_sine_sequence(length, periods=3):
    t = np.arange(length)
    omega = 2 * np.pi * periods / length
    return np.sin(omega * t).reshape(-1, 1)

In [ ]:
hidden_size = 1
lr = 1e-2
epochs = 60
seq_len = 100
batch_size = 128

model = TrainableLSTM(input_size=1, hidden_size=hidden_size, output_size=1, lr=lr)

losses = []
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0
    for _ in range(batch_size):
        seq = make_sine_sequence(seq_len + 1, periods=3)
        x = seq[:-1]
        y_true = seq[1:]

        y_pred, _, caches = model.forward_sequence(x, store_cache=True)

        err = y_pred - y_true.reshape(seq_len, 1, 1)
        loss = np.mean(err ** 2)
        epoch_loss += loss

        dy = (2.0 / seq_len) * err
        model.backward_sequence(dy, caches, clip=1.0)

    epoch_loss /= batch_size
    losses.append(epoch_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{epochs} | Loss: {epoch_loss:.8f}")

print("\nTraining complete.")

# Teacher-forced test
test_seq = make_sine_sequence(seq_len + 1, periods=3)
x_test = test_seq[:-1]
y_test = test_seq[1:]

y_test_pred, _, _ = model.forward_sequence(x_test, store_cache=False)
y_pred_arr = y_test_pred.reshape(seq_len, 1)
y_true_arr = y_test.reshape(seq_len, 1)

print("\nTeacher-forced next-step predictions (first 10):")
for t in range(10):
    print(f"input={x_test[t,0]:7.4f} -> pred_next={y_pred_arr[t,0]:8.4f} | true_next={y_true_arr[t,0]:7.4f}")

# Autoregressive rollout from one starting value
h_roll = np.zeros((hidden_size, 1))
c_roll = np.zeros((hidden_size, 1))
x_cur = np.array([[0.0]])

generated = [float(x_cur[0, 0])]
for _ in range(99):
    y_one, (h_roll, c_roll), _ = model.forward_sequence(x_cur, h_roll, c_roll, store_cache=False)
    x_cur = y_one[-1].reshape(1, 1)
    generated.append(float(x_cur[0, 0]))

print("\nAutoregressive rollout (first 20 steps):")
print([round(v, 4) for v in generated[:20]])

In [ ]:
# Compare custom NumPy LSTM with Keras LSTM
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# Generate a sine sequence dataset for Keras
seq_len = 100
batch_size = 128
periods = 3
X = []
y = []
for _ in range(batch_size):
    seq = np.sin(np.linspace(0, 2 * np.pi * periods, seq_len + 1)).reshape(-1, 1)
    X.append(seq[:-1])
    y.append(seq[1:])
X = np.stack(X)  # shape: (batch_size, seq_len, 1)
y = np.stack(y)  # shape: (batch_size, seq_len, 1)

# Keras LSTM model
keras_model = Sequential([
    LSTM(8, input_shape=(seq_len, 1), return_sequences=True),
    Dense(1)
])
keras_model.compile(optimizer=Adam(1e-2), loss='mse')
keras_model.fit(X, y, epochs=60, batch_size=batch_size, verbose=2)

# Predict with Keras LSTM
keras_pred = keras_model.predict(X[0:1])
print("Keras LSTM predictions (first 10):")
print(np.round(keras_pred[0, :10, 0], 4))

# Predict with custom LSTM
custom_model = TrainableLSTM(input_size=1, hidden_size=8, output_size=1, lr=1e-2)
for epoch in range(60):
    for i in range(batch_size):
        x = X[i]
        y_true = y[i]
        y_pred, _, caches = custom_model.forward_sequence(x, store_cache=True)
        err = y_pred - y_true.reshape(seq_len, 1, 1)
        dy = (2.0 / seq_len) * err
        custom_model.backward_sequence(dy, caches, clip=1.0)
custom_pred, _, _ = custom_model.forward_sequence(X[0], store_cache=False)
print("Custom LSTM predictions (first 10):")
print(np.round(custom_pred[:10, 0, 0], 4))

Epoch 1/60


C:\Users\sbrad\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 - 2s - 2s/step - loss: 0.5229
Epoch 2/60
1/1 - 0s - 68ms/step - loss: 0.4773
Epoch 3/60
1/1 - 0s - 69ms/step - loss: 0.4356
Epoch 4/60
1/1 - 0s - 56ms/step - loss: 0.3965
Epoch 5/60
1/1 - 0s - 59ms/step - loss: 0.3594
Epoch 6/60
1/1 - 0s - 59ms/step - loss: 0.3241
Epoch 7/60
1/1 - 0s - 59ms/step - loss: 0.2904
Epoch 8/60
1/1 - 0s - 58ms/step - loss: 0.2584
Epoch 9/60
1/1 - 0s - 56ms/step - loss: 0.2278
Epoch 10/60
1/1 - 0s - 64ms/step - loss: 0.1983
Epoch 11/60
1/1 - 0s - 66ms/step - loss: 0.1696
Epoch 12/60
1/1 - 0s - 82ms/step - loss: 0.1417
Epoch 13/60
1/1 - 0s - 58ms/step - loss: 0.1145
Epoch 14/60
1/1 - 0s - 65ms/step - loss: 0.0887
Epoch 15/60
1/1 - 0s - 83ms/step - loss: 0.0649
Epoch 16/60
1/1 - 0s - 59ms/step - loss: 0.0441
Epoch 17/60
1/1 - 0s - 94ms/step - loss: 0.0272
Epoch 18/60
1/1 - 0s - 63ms/step - loss: 0.0152
Epoch 19/60
1/1 - 0s - 60ms/step - loss: 0.0086
Epoch 20/60
1/1 - 0s - 52ms/step - loss: 0.0073
Epoch 21/60
1/1 - 0s - 54ms/step - loss: 0.0102
Epoch 22/60
1/

In [21]:
y_true.reshape(seq_len, 1, 1)[:10, 0, 0]

array([0.18738131, 0.36812455, 0.53582679, 0.68454711, 0.80901699,
       0.90482705, 0.96858316, 0.99802673, 0.9921147 , 0.95105652])